# Notebook 4: ProtT5 + LoRA Fine-Tuning

**Prerequisite:** run `python run_preprocessing.py` first.

Fine-tunes ProtT5-XL-UniRef50 encoder with LoRA adapters (or QLoRA for low-VRAM GPUs).

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import torch
import torch.nn as nn
import yaml
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import DataLoader

from models.prott5_classifier import build_prott5_classifier, ProtT5Classifier
from dataset import ProtT5Dataset, collate_fn_pad
from evaluate import compute_fmax, compute_aupr, compute_coverage, compute_smin, load_ia_weights
from evaluate import compute_precision_recall_curve
from preprocessing import load_processed

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OBO_PATH = '../data/raw/Train/go-basic.obo'
IA_PATH  = '../data/raw/IA.tsv'
print(f'Device: {DEVICE}')

## 1. Configuration

In [ ]:
with open('../config/prott5_lora.yaml') as f:
    cfg = yaml.safe_load(f)

# Reduce for quick testing; restore full values for submission training
cfg['training']['num_epochs']   = 8
cfg['training']['batch_size']   = 2
cfg['training']['gradient_accumulation_steps'] = 16   # effective batch = 32

# Enable QLoRA for ≤16 GB VRAM GPUs
# cfg['qlora']['enabled'] = True

ONTOLOGY   = 'MFO'
DATA_PATH  = f'../data/processed/cafa6_{ONTOLOGY.lower()}.pkl'
OUTPUT_DIR = f'../outputs/prott5_lora_{ONTOLOGY.lower()}'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f'Ontology: {ONTOLOGY} | Base model: {cfg["model"]["base_model"]}')

## 2. Load Data

In [ ]:
data       = load_processed(DATA_PATH)
go_terms   = data['go_terms']
num_labels = len(go_terms)
ia_weights = load_ia_weights(IA_PATH, go_terms)
print(f'GO terms ({ONTOLOGY}): {num_labels:,}')

tokenizer = AutoTokenizer.from_pretrained(cfg['model']['base_model'])
max_len   = cfg['model']['max_length']

def make_loader(split_key, shuffle, bs=None):
    split = data[split_key]
    bs = bs or cfg['training']['batch_size']
    ds = ProtT5Dataset(split['sequences'], tokenizer, max_len, split['labels'], split['protein_ids'])
    return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                      collate_fn=collate_fn_pad, num_workers=0, pin_memory=(DEVICE.type=='cuda'))

train_loader = make_loader('train', shuffle=True)
val_loader   = make_loader('val',   shuffle=False, bs=4)
test_loader  = make_loader('test',  shuffle=False, bs=4)
print(f'Train batches: {len(train_loader)}')

## 3. Build ProtT5 + LoRA

In [ ]:
model = build_prott5_classifier(cfg, num_labels, go_terms).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## 4. Training

In [ ]:
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=cfg['training']['learning_rate'],
    weight_decay=cfg['training']['weight_decay'],
)
num_epochs   = cfg['training']['num_epochs']
grad_accum   = cfg['training']['gradient_accumulation_steps']
total_steps  = (len(train_loader) // grad_accum) * num_epochs
warmup_steps = int(total_steps * cfg['training']['warmup_ratio'])
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

use_bf16 = cfg['training'].get('bf16', False) and DEVICE.type == 'cuda'
use_fp16 = cfg['training'].get('fp16', False) and DEVICE.type == 'cuda'
scaler   = torch.cuda.amp.GradScaler() if use_fp16 else None

history = {'train_loss': [], 'val_loss': [], 'fmax': [], 'aupr': []}

@torch.no_grad()
def eval_loader(loader, mdl=None):
    mdl = mdl or model
    mdl.eval()
    logits_l, labels_l = [], []
    total_loss = 0.0
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        labs = batch['labels'].to(DEVICE)
        out  = mdl(ids, mask, labs)
        total_loss += out['loss'].item()
        logits_l.append(out['logits'].cpu().float().numpy())
        labels_l.append(labs.cpu().numpy())
    scores = torch.sigmoid(torch.tensor(np.concatenate(logits_l))).numpy()
    y_true = np.concatenate(labels_l)
    return y_true, scores, total_loss / len(loader)

best_fmax = 0.0
best_ckpt = f'{OUTPUT_DIR}/best'

for epoch in range(1, num_epochs + 1):
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(train_loader, desc=f'Epoch {epoch}/{num_epochs}')):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        labs = batch['labels'].to(DEVICE)

        amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
        with torch.autocast(device_type=DEVICE.type, dtype=amp_dtype,
                            enabled=(use_bf16 or use_fp16)):
            out  = model(ids, mask, labs)
            loss = out['loss'] / grad_accum

        (scaler.scale(loss) if scaler else loss).backward()
        epoch_loss += loss.item() * grad_accum

        if (step + 1) % grad_accum == 0:
            if scaler:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    y_true, scores, val_loss = eval_loader(val_loader)
    fmax, best_t = compute_fmax(y_true, scores)
    aupr = compute_aupr(y_true, scores)

    tl = epoch_loss / len(train_loader)
    history['train_loss'].append(tl)
    history['val_loss'].append(val_loss)
    history['fmax'].append(fmax)
    history['aupr'].append(aupr)

    print(f'Epoch {epoch:02d} | train={tl:.4f} | val={val_loss:.4f} | Fmax={fmax:.4f} | AUPR={aupr:.4f}')

    if fmax > best_fmax:
        best_fmax = fmax
        model.save_pretrained(best_ckpt)
        print(f'  >>> Best Fmax={fmax:.4f}')

## 5. Test Evaluation + PR Curve

In [ ]:
best_model = ProtT5Classifier.from_pretrained(best_ckpt).to(DEVICE)
y_true_ts, scores_ts, _ = eval_loader(test_loader, best_model)

fmax_ts, t_ts  = compute_fmax(y_true_ts, scores_ts)
aupr_ts  = compute_aupr(y_true_ts, scores_ts)
cov_ts   = compute_coverage(scores_ts, t_ts)
smin_ts, _ = compute_smin(y_true_ts, scores_ts, ia_weights)

print(f'=== Test ({ONTOLOGY}) ===')
print(f'Fmax     : {fmax_ts:.4f}  @ t={t_ts:.2f}')
print(f'AUPR     : {aupr_ts:.4f}')
print(f'Smin     : {smin_ts:.4f}')
print(f'Coverage : {cov_ts:.4f}')

prec, rec, _ = compute_precision_recall_curve(y_true_ts, scores_ts)
plt.figure(figsize=(6, 5))
plt.plot(rec, prec, lw=2, color='steelblue')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title(f'ProtT5 LoRA — {ONTOLOGY} | Fmax={fmax_ts:.3f} AUPR={aupr_ts:.3f}')
plt.tight_layout()
plt.savefig(f'../reports/prott5_{ONTOLOGY.lower()}_pr_curve.png', dpi=150)
plt.show()

## 6. Push to HuggingFace Hub

In [ ]:
HUB_REPO = 'your-username/cafa6-prott5-lora-mfo'   # <-- change this
HF_TOKEN = 'hf_...'                                  # <-- or run: huggingface-cli login

# Option A: push LoRA adapters only
# best_model.push_to_hub(HUB_REPO, token=HF_TOKEN)

# Option B: merge LoRA into base, push full model
# best_model.merge_and_push(HUB_REPO, token=HF_TOKEN)

print('Set HUB_REPO and HF_TOKEN, then uncomment above.')